# Ödev 

### 1. Tokenizer Detayları ve Padding Analizi (Başlangıç Seviye)

Hugging Face `AutoTokenizer` kullanarak farklı uzunluktaki en az 5 Türkçe cümleyi aynı anda işleyin.
- `padding=True` ve `truncation=True` parametrelerinin etkisini inceleyin.
- Oluşan `attention_mask` değerlerini görselleştirin (veya yazdırın) ve modelin neden bazı tokenlara "0" verdiğini açıklayın.
- Özel tokenların ([CLS], [SEP], [PAD]) ID'lerini ve metindeki yerlerini tespit edin.

*Kütüphaneler: transformers, torch*

In [11]:
from transformers import AutoTokenizer

model_checkpoint = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

cumleler = [
    "Merhaba.",
    "Bugün dersimizde tokenizer detaylarını işliyoruz.",
    "Yapay zeka projelerinde doğru tokenizasyon, modelin bağlamı anlaması açısından kritik bir adımdır.",
    "Kısa cümle.",
    "Bu cümle özellikle biraz daha uzun tutuldu çünkü truncation=True etkisini net görmek istiyorum ve maksimum uzunluğu bilinçli olarak sınırlandırıyorum."
]

inputs = tokenizer(
    cumleler,
    padding=True,
    truncation=True,
    max_length=24,
    return_tensors="pt"
)

print("input_ids boyutu:", tuple(inputs["input_ids"].shape))
print("attention_mask boyutu:", tuple(inputs["attention_mask"].shape))

print("\ninput_ids:")
print(inputs["input_ids"])

print("\nattention_mask:")
print(inputs["attention_mask"])

print("\nNeden bazı tokenlar 0?")
print("attention_mask=1 olanlar gerçek token, 0 olanlar padding tokenlarıdır (model onları dikkate almaz).")

cls_id = tokenizer.cls_token_id
sep_id = tokenizer.sep_token_id
pad_id = tokenizer.pad_token_id

print("\nÖzel token ID'leri:")
print(f"[CLS]: {cls_id}, [SEP]: {sep_id}, [PAD]: {pad_id}")

for i, ids in enumerate(inputs["input_ids"]):
    ids_list = ids.tolist()
    cls_pos = [idx for idx, v in enumerate(ids_list) if v == cls_id]
    sep_pos = [idx for idx, v in enumerate(ids_list) if v == sep_id]
    pad_pos = [idx for idx, v in enumerate(ids_list) if v == pad_id]

    print(f"\nCümle {i+1} token konumları:")
    print("[CLS] konumu:", cls_pos)
    print("[SEP] konumu:", sep_pos)
    print("[PAD] konumları:", pad_pos)

print("\nİlk cümlenin token karşılıkları:")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

input_ids boyutu: (5, 24)
attention_mask boyutu: (5, 24)

input_ids:
tensor([[    2,  6965,    18,     3,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    2,  4148, 10653,  2938,  1050,  1988,  8119, 11481,  1028, 26288,
         29046,  2086,    18,     3,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    2, 27980, 14348, 31231,  2713,  8119,  2741,  2510,    16, 23700,
          9959,  1048,  6289,  2499,  4122, 10587,  1996,  3816,  2077,    18,
             3,     0,     0,     0],
        [    2,  8147,  9446,    18,     3,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [    2,  2123,  9446,  3550,  3165,  2171,  2956,  8291,  2045,  3845,
          3307,  3170,  6403,    33, 18694,  1025, 12198,  3984,  46

### 2. Model Mimarileri ve Pipeline Karşılaştırması (Orta Seviye)

Üç farklı görev için Hugging Face Hub'dan uygun modeller seçin:
1. **Duygu Analizi** (Encoder-only - örn: BERT)
2. **Metin Özetleme** (Encoder-Decoder - örn: T5 veya BART)
3. **Metin Üretimi** (Decoder-only - örn: GPT-2 veya TinyLlama)

- Aynı girdi metniyle (veya göreve uygun varyasyonlarıyla) bu modelleri test edin.
- Her mimarinin neden o görev için daha uygun olduğunu (1. haftadaki teorik bilgilerle birleştirerek) kısa notlar halinde hücre altına yazın.

In [12]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch

metin = (
    "Yapay zeka tabanli cozumler saglik, egitim ve ulasim gibi alanlarda verimliligi artiriyor. "
    "Ama etik kullanim ve veri guvenligi hala onemli bir konu."
)

print("1) Duygu analizi")
try:
    duygu_modeli = "savasy/bert-base-turkish-sentiment-cased"
    duygu_pipe = pipeline("sentiment-analysis", model=duygu_modeli)
    print(duygu_pipe(metin))
except Exception as e:
    print("Duygu analizi hatasi:", e)

print("\n2) Ozetleme")
try:
    ozet_modeli = "google/flan-t5-small"
    ozet_tokenizer = AutoTokenizer.from_pretrained(ozet_modeli)
    ozet_model = AutoModelForSeq2SeqLM.from_pretrained(ozet_modeli)
    ozet_istek = f"Bu metni kisa ozetle: {metin}"
    girdiler = ozet_tokenizer(ozet_istek, return_tensors="pt", truncation=True)
    with torch.no_grad():
        cikti = ozet_model.generate(**girdiler, max_new_tokens=50)
    print(ozet_tokenizer.decode(cikti[0], skip_special_tokens=True))
except Exception as e:
    print("Ozetleme hatasi:", e)

print("\n3) Metin uretimi")
try:
    uretim_modeli = "sshleifer/tiny-gpt2"
    uretim_pipe = pipeline("text-generation", model=uretim_modeli)
    uretim_pipe.tokenizer.pad_token = uretim_pipe.tokenizer.eos_token
    uretim_pipe.model.config.pad_token_id = uretim_pipe.tokenizer.eos_token_id
    sonuc = uretim_pipe("Yapay zeka ile gelecegin sehirleri", max_new_tokens=40, do_sample=True, temperature=0.8)
    print(sonuc[0]["generated_text"])
except Exception as e:
    print("Uretim hatasi:", e)

print("\nKisa not:")
print("- Encoder-only yapilar siniflandirmada daha islevsel.")
print("- Encoder-decoder yapilar ozetleme gibi donusum islerinde daha uygun.")
print("- Decoder-only yapilar serbest metin uretiminde iyi calisiyor.")

1) Duygu analizi


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4896.50it/s]


[{'label': 'negative', 'score': 0.959404706954956}]

2) Ozetleme


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 3836.87it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


This is a very important step: The re-election of the armed forces, the armed forces and the armed forces are a step in the right direction.

3) Metin uretimi


Loading weights: 100%|██████████| 29/29 [00:00<00:00, 16170.54it/s]
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yapay zeka ile gelecegin sehirlerimediately substimura scalpdit ESV subst ONE004 TA Danielting004 vendors Jr reborn reborn Prob directly TAdit pawn Participation Participation trilogy Observ Jr scalp Probmediately confir Participationting HancockScene ESV antibioticmediatelymediately Prob

Kisa not:
- Encoder-only yapilar siniflandirmada daha islevsel.
- Encoder-decoder yapilar ozetleme gibi donusum islerinde daha uygun.
- Decoder-only yapilar serbest metin uretiminde iyi calisiyor.


### 3. Veri Seti Manipülasyonu ve Temizliği (Orta Seviye)

Hugging Face `datasets` kütüphanesini kullanarak `savasy/ttc4900` veya benzeri bir Türkçe veri seti yükleyin.
- `.map()` fonksiyonunu kullanarak tüm metinleri küçük harfe çevirin ve noktalama işaretlerini kaldırın.
- `.filter()` fonksiyonu ile belirli bir anahtar kelimeyi (örn: "teknoloji") içeren metinleri ayıklayın.
- Veri setini %80 eğitim, %20 test olacak şekilde bölün.

In [13]:
from datasets import load_dataset
import re

veri = load_dataset("savasy/ttc4900", split="train")

if "text" in veri.column_names:
    metin_alani = "text"
else:
    metin_alani = veri.column_names[0]


def temizle(kayit):
    metin = kayit[metin_alani].lower()
    metin = re.sub(r"[^\w\s]", "", metin)
    metin = re.sub(r"\s+", " ", metin).strip()
    kayit[metin_alani] = metin
    return kayit

veri_temiz = veri.map(temizle)
veri_filtre = veri_temiz.filter(lambda x: "teknoloji" in x[metin_alani])

print("Kullanilan alan:", metin_alani)
print("Temiz veri sayisi:", len(veri_temiz))
print("Filtreli veri sayisi:", len(veri_filtre))

if len(veri_filtre) == 0:
    print("Bu kelimeyle sonuc cikmadi.")
else:
    bolunmus = veri_filtre.train_test_split(test_size=0.2, seed=42)
    print("Train:", len(bolunmus["train"]))
    print("Test:", len(bolunmus["test"]))
    print("\nOrnek:")
    print(bolunmus["train"][0])

Filter: 100%|██████████| 4900/4900 [00:00<00:00, 50455.50 examples/s]


Kullanilan alan: text
Temiz veri sayisi: 4900
Filtreli veri sayisi: 380
Train: 304
Test: 76

Ornek:
{'category': 4, 'text': 'hayalinizdeki vücuda ameliyatsız kavuşmak ister misiniz tüm kadınlar ince bir vücudun hayalini kurar eski yöntemlerle yapıldığında günlerce dinlenme gerektiren estetik operasyonlar artık ağrısız ve acısız bir şekilde kolayca uygulanabiliyor yeni teknolojiyle geliştirilen yöntemler vücut şekillendirmede estetik cerrahi işlemlerinin yönünü değiştiriyor superplast estetik cerrahi merkezi nden estetik plastik ve rekonstrüktif_cerrahi_uzmanı_op dr coşkun levent taşçı ameliyatsız vücut şekillendirmeyle ilgili şu önemli bilgileri aktarıyor artık estetik cerrahları elinde kanlı bisturi kocaman doku makası ve kocaman bir çelik liposuction çubuğu ile ameliyathanelerde görmüyoruz bunun yerine küçücük fiber optik kanüller veya bilgisayarlar benzeri gelişmiş lazer cihazları ile yapılıyor bu mucize benzeri teknoloji destekli estetik cerrahi işlemler lazer destekli estetik cerr

### 3. Veri Seti Manipülasyonu ve Temizliği (Orta Seviye)

Hugging Face `datasets` kütüphanesini kullanarak `savasy/ttc4900` veya benzeri bir Türkçe veri seti yükleyin.
- `.map()` fonksiyonunu kullanarak tüm metinleri küçük harfe çevirin ve noktalama işaretlerini kaldırın.
- `.filter()` fonksiyonu ile belirli bir anahtar kelimeyi (örn: "teknoloji") içeren metinleri ayıklayın.
- Veri setini %80 eğitim, %20 test olacak şekilde bölün.

In [14]:
from datasets import load_dataset
import re

veri = load_dataset("savasy/ttc4900", split="train")
alan = "text" if "text" in veri.column_names else veri.column_names[0]


def normalize(kayit):
    metin = kayit[alan].lower()
    metin = re.sub(r"[^\w\s]", "", metin)
    kayit[alan] = re.sub(r"\s+", " ", metin).strip()
    return kayit

veri = veri.map(normalize)
veri = veri.filter(lambda x: "teknoloji" in x[alan])

print("Kullanilan alan:", alan)
if len(veri) == 0:
    print("Filtre sonucu bos. Farkli kelime deneyebilirsin.")
else:
    parcalar = veri.train_test_split(test_size=0.2, seed=42)
    print("Toplam:", len(parcalar["train"]) + len(parcalar["test"]))
    print("Egitim:", len(parcalar["train"]))
    print("Test:", len(parcalar["test"]))
    print("\nIlk 2 ornek:")
    for i in range(min(2, len(parcalar["train"]))):
        print(f"- {parcalar['train'][i][alan][:140]}...")

Filter: 100%|██████████| 4900/4900 [00:00<00:00, 64092.29 examples/s]

Kullanilan alan: text
Toplam: 380
Egitim: 304
Test: 76

Ilk 2 ornek:
- hayalinizdeki vücuda ameliyatsız kavuşmak ister misiniz tüm kadınlar ince bir vücudun hayalini kurar eski yöntemlerle yapıldığında günlerce ...
- chp nin erdoğan hakkında verdiği önerge görüşülecek meclis chp nin başbakan_recep_tayyip_erdoğan hakkında verdiği soruşturma önergesi ile 20...


### 4. Quantization ve Bellek Analizi (İleri Seviye)

Bir LLM modelini (örn: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`) iki farklı şekilde yükleyin:
1. Normal (FP16/FP32) hassasiyette.
2. 4-bit Quantization (`bitsandbytes` kullanarak) ile.

- Her iki yükleme yönteminde modelin GPU/RAM üzerinde kapladığı alanı karşılaştırın.
- Aynı prompt için üretim hızlarını (tokens per second) kabaca ölçün.
- *Not: Bu ödev için Google Colab T4 GPU kullanılması önerilir.*

In [15]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
prompt = "Yapay zekanin egitimdeki etkisini 3 maddede acikla."


def model_bellek_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)


def hiz_olc(model, tokenizer, prompt_text, device):
    giris = tokenizer(prompt_text, return_tensors="pt").to(device)
    baslangic = time.perf_counter()
    with torch.no_grad():
        cikti = model.generate(**giris, max_new_tokens=64, do_sample=False)
    sure = time.perf_counter() - baslangic
    yeni_token = cikti.shape[1] - giris["input_ids"].shape[1]
    tps = yeni_token / sure if sure > 0 else 0.0
    return sure, tps

cihaz = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", cihaz)

if cihaz != "cuda":
    print("Bu test GPU icin daha uygun. CPU'da cok yavas olacagi icin atliyorum.")
    print("Istersen Colab T4 ile calistirabilirsin.")
else:
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    normal_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    normal_bellek = model_bellek_mb(normal_model)
    normal_sure, normal_tps = hiz_olc(normal_model, tokenizer, prompt, "cuda")

    print("\nNormal yukleme")
    print(f"- Tahmini model bellegi: {normal_bellek:.2f} MB")
    print(f"- Uretim suresi: {normal_sure:.2f} sn")
    print(f"- Hiz: {normal_tps:.2f} token/sn")

    try:
        q_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )

        quant_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=q_config,
            device_map="auto"
        )

        quant_bellek = model_bellek_mb(quant_model)
        quant_sure, quant_tps = hiz_olc(quant_model, tokenizer, prompt, "cuda")

        print("\n4-bit quantization")
        print(f"- Tahmini model bellegi: {quant_bellek:.2f} MB")
        print(f"- Uretim suresi: {quant_sure:.2f} sn")
        print(f"- Hiz: {quant_tps:.2f} token/sn")

        print("\nKarsilastirma")
        print(f"- Bellek orani: {normal_bellek / quant_bellek:.2f}x")
        print(f"- Hiz orani: {normal_tps / quant_tps:.2f}x")
    except Exception as e:
        print("\n4-bit kismi bu ortamda calismadi.")
        print("Hata:", e)
        print("Bu bolum Colab T4'te daha stabil.")

Cihaz: cpu
Bu test GPU icin daha uygun. CPU'da cok yavas olacagi icin atliyorum.
Istersen Colab T4 ile calistirabilirsin.
